# EMASEO -- Entrenamiento RT-DETR-L en Google Colab T4
**Modelo:** RT-DETR-L (Real-Time Detection Transformer, Large)
**Objetivo:** mAP@50 >= 0.85 | nc=1 | clase: garbage
**Baseline actual:** mAP@50 = 0.4752
**Dataset:** 21,987 train (21,486 con basura + 501 backgrounds Quito) / 3,531 valid

## Tiempo estimado de entrenamiento
| Escenario | Tiempo |
|-----------|--------|
| Convergencia temprana (epoca ~50) | ~14 horas |
| Convergencia normal (epoca ~70) | ~20 horas |
| Maximo (100 epocas) | ~28 horas |

**Estrategia:** entrenamos en 2 sesiones de Colab usando resume=True.
Sesion 1: epocas 1-50 (~14h). Sesion 2: continuar desde last.pt.

## Antes de empezar
1. Menu superior: **Runtime > Change runtime type > T4 GPU**
2. Sube `dataset_entrenamiento.zip` a Google Drive (carpeta raiz `Mi unidad/`)
3. Ejecuta las celdas en orden

## Celda 1 -- Verificar GPU

In [ ]:
!nvidia-smi
import torch
print(f'CUDA disponible: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Celda 2 -- Instalar dependencias

In [ ]:
!pip install ultralytics==8.3.* -q
from ultralytics import RTDETR
import ultralytics
print(f'Ultralytics {ultralytics.__version__} listo')

## Celda 3 -- Montar Google Drive y extraer dataset

In [ ]:
from google.colab import drive
from pathlib import Path
import os, zipfile

drive.mount('/content/drive')

ZIP_PATH    = '/content/drive/MyDrive/dataset_entrenamiento.zip'
DATASET_DIR = '/content/dataset_entrenamiento'

if not os.path.exists(DATASET_DIR):
    print('Extrayendo dataset...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    print('Listo')
else:
    print('Dataset ya extraido')

for split in ['train', 'valid']:
    p = Path(f'{DATASET_DIR}/{split}/images')
    imgs = list(p.glob('*')) if p.exists() else []
    labels = list((Path(f'{DATASET_DIR}/{split}/labels')).glob('*.txt')) if (Path(f'{DATASET_DIR}/{split}/labels')).exists() else []
    print(f'  {split}: {len(imgs):,} imagenes | {len(labels):,} labels | {len(imgs)-len(labels):,} backgrounds')

## Celda 4 -- Fijar data.yaml con paths absolutos de Colab

In [ ]:
from pathlib import Path

DATASET_DIR = '/content/dataset_entrenamiento'
YAML_PATH   = f'{DATASET_DIR}/data.yaml'

yaml_content = f"""path: {DATASET_DIR}
train: train/images
val:   valid/images
test:  test/images
nc: 1
names:
  - garbage
"""
Path(YAML_PATH).write_text(yaml_content)
print('data.yaml OK')
print(Path(YAML_PATH).read_text())

## Celda 5 -- ENTRENAMIENTO (SESION 1: epocas 1-100)

> **Tiempo estimado: 12-20 horas segun cuando haga early stopping.**
>
> Estrategia de 2 sesiones:
> - Sesion 1 (esta celda): entrena hasta que Colab se agote o llegue a 100 epocas
> - Sesion 2 (Celda 7): retoma desde last.pt con resume=True
>
> El modelo se guarda en Drive cada 5 epocas. Peor caso: pierdes 5 epocas.
>
> **Por que AMP=True en T4 y no en RTX 3050:**
> T4 (Turing) soporta FP16 estable. RTX 3050 (Ampere) tiene bug conocido
> con RT-DETR que produce NaN. En Colab T4 funciona perfectamente.

In [ ]:
from ultralytics import RTDETR

model = RTDETR('rtdetr-l.pt')

results = model.train(
    data='/content/dataset_entrenamiento/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    patience=25,       # early stopping: detiene si no mejora 25 epocas seguidas
    optimizer='AdamW',
    lr0=0.0001,
    lrf=0.01,
    warmup_epochs=5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    flipud=0.3,
    fliplr=0.5,
    mosaic=1.0,
    copy_paste=0.1,
    project='/content/drive/MyDrive/emaseo_runs',
    name='rtdetr_l_emaseo_v2',
    exist_ok=True,
    save=True,
    save_period=5,     # checkpoint cada 5 epocas (no 10) para no perder progreso
    amp=True,
    verbose=True,
)

map50 = results.results_dict.get('metrics/mAP50(B)', 0)
print(f'\nmAP@50 al final sesion 1: {map50:.4f}')
print(f'Baseline previo:          0.4752')
print(f'Mejora:                   {map50 - 0.4752:+.4f}')
print(f'\nbest.pt guardado en: /content/drive/MyDrive/emaseo_runs/rtdetr_l_emaseo_v2/weights/best.pt')

## Celda 6 -- Validacion final con best.pt

In [ ]:
from ultralytics import RTDETR

best_path = '/content/drive/MyDrive/emaseo_runs/rtdetr_l_emaseo_v2/weights/best.pt'
model = RTDETR(best_path)

metrics = model.val(
    data='/content/dataset_entrenamiento/data.yaml',
    imgsz=640,
    batch=16,
    device=0,
)

map50 = metrics.results_dict['metrics/mAP50(B)']
prec  = metrics.results_dict['metrics/precision(B)']
rec   = metrics.results_dict['metrics/recall(B)']

print(f'mAP@50:    {map50:.4f}  (baseline: 0.4752)')
print(f'Precision: {prec:.4f}')
print(f'Recall:    {rec:.4f}')
print()
if map50 >= 0.85:
    print('OBJETIVO ALCANZADO -- listo para produccion')
elif map50 >= 0.75:
    print(f'Buen resultado. Fine-tuning con fotos aereas de Quito llevara al 0.85')
else:
    print(f'Continuar entrenamiento con Celda 7 (resume=True)')

## Celda 7 -- Reanudar entrenamiento (SESION 2 o si Colab se desconecto)

> Ejecutar si:
> - Colab se desconecto durante la Sesion 1
> - El mAP@50 de Celda 6 fue < 0.80 y quieres mas epocas
>
> **Antes de ejecutar:** ejecuta Celdas 1, 2 y 3 nuevamente para reinstalar
> dependencias y re-montar Drive.

In [ ]:
from ultralytics import RTDETR

last_checkpoint = '/content/drive/MyDrive/emaseo_runs/rtdetr_l_emaseo_v2/weights/last.pt'
model = RTDETR(last_checkpoint)

# resume=True retoma desde la epoca donde quedo, con el mismo lr y configuracion
results = model.train(resume=True)

map50 = results.results_dict.get('metrics/mAP50(B)', 0)
print(f'mAP@50 final: {map50:.4f}')

## Celda 8 -- Descargar best.pt a tu PC

In [ ]:
# best.pt esta en Drive: Mi unidad/emaseo_runs/rtdetr_l_emaseo_v2/weights/best.pt
# Descarga directa:
from google.colab import files
files.download('/content/drive/MyDrive/emaseo_runs/rtdetr_l_emaseo_v2/weights/best.pt')